In [1]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 0
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate_hkqai_done").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict_ccpvdz"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099

        data_error_scf_ele = data["error_scf_ele"].to_numpy()
        data_error_dft_ele = data["error_dft_ele"].to_numpy()
        data_error_scf_dip = data["error_scf_dip"].to_numpy()
        data_error_dft_dip = data["error_dft_dip"].to_numpy()
        data_subset[f"{data_path_name}_summary"] = {
            "error_scf_ele": data_error_scf_ele,
            "error_dft_ele": data_error_dft_ele,
            "error_scf_dip": data_error_scf_dip,
            "error_dft_dip": data_error_dft_dip,
        }

        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if "modified_dft_d3bj" in data.columns:
            data_dft_d3bj = data["modified_dft_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3bj = data_d3bj
        if "modified_dft_d3zero" in data.columns:
            data_dft_d3zero = data["modified_dft_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3zero = data_d3zero
        if "modified_ai_d3bj" in data.columns:
            data_ai_d3bj = data["modified_ai_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3bj = data_d3bj
        if "modified_ai_d3zero" in data.columns:
            data_ai_d3zero = data["modified_ai_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
                "error_scf_ele": [],
                "error_dft_ele": [],
                "error_scf_dip": [],
                "error_dft_dip": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(
                            f"Warning: {i_molecule_name} not found in {data_path.stem} data file"
                        )
                    continue
                data_subset[name_subset]["error_scf_ele"].append(
                    data_error_scf_ele[col[0]]
                )
                data_subset[name_subset]["error_dft_ele"].append(
                    data_error_dft_ele[col[0]]
                )
                data_subset[name_subset]["error_scf_dip"].append(
                    data_error_scf_dip[col[0]]
                )
                data_subset[name_subset]["error_dft_dip"].append(
                    data_error_dft_dip[col[0]]
                )

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    print(atomic_energy_cc, i_reaction["reference"])
                    atomic_energy_cc = i_reaction["reference"]
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_ai = np.argsort(data_subset[name_subset]["ai"])[
                    ::-1
                ][:5]
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ][:5]
                for i in range(len(argsort_atomic_energy_ai)):
                    i_reaction_name = data_subset[name_subset]["name"][
                        argsort_atomic_energy_ai[i]
                    ]
                    i_reaction = json_data[f"reaction-{i_subset}"][i_reaction_name]
                    print(
                        f"Top {i+1} AI: {data_subset[name_subset]['ai'][argsort_atomic_energy_ai[i]]} kcal/mol, {i_reaction_name} in {name_subset}",
                    )
                    systems_list = i_reaction["systems"]
                    stoichiometry_list = i_reaction["stoichiometry"]
                    for j in range(len(systems_list)):
                        mole_name = (
                            systems_list[j]
                            if i_subset == "BH76RC"
                            else f"{i_subset}-{systems_list[j]}"
                        )
                        stoichiometry = int(stoichiometry_list[j])
                        if mole_name in json_data:
                            if isinstance(json_data[mole_name], str):
                                mole_name = json_data[mole_name]
                        print(f"  {stoichiometry} * {mole_name}", end="")
                        col = np.where(data_name == mole_name)[0]
                        if col.size == 1:
                            error_energy_ai = data_scf[col[0]] - data_cc[col[0]]
                            print(f"  {stoichiometry} * {error_energy_ai}", end="")
                    print()

                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

df_summary_subset_ele = pd.DataFrame(
    columns=pd.MultiIndex.from_product(
        [
            data_path_name_list,
            [
                "error_scf_ele",
                "error_dft_ele",
                "error_scf_dip",
                "error_dft_dip",
            ],
        ],
        names=["data_path", "Ele type"],
    )
)

for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_dip"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_dip"]
    )

    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}

        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_ele")] = (
                np.mean(data_subset[name_subset]["error_scf_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_ele")] = (
                np.mean(data_subset[name_subset]["error_dft_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_dip")] = (
                np.mean(data_subset[name_subset]["error_scf_dip"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_dip")] = (
                np.mean(data_subset[name_subset]["error_dft_dip"])
            )

            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    "DONE"
                    if (
                        (
                            len(data_subset[name_subset]["ai"])
                            == len(data_subset[name_subset]["name"])
                        )
                        and (len(data_subset[name_subset]["ai"]) != 0)
                    )
                    else f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    print(
        f"Mean absolute deviation for {data_path_name}: {mean_absolute_deviation:.4f} kcal/mol"
    )
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

    wtmad_1_subset.loc["summary", (data_path_name, "Processed")] = "--"
    wtmad_2_subset.loc["summary", (data_path_name, "Processed")] = "--"
    for d3_name in ["", "_d3bj", "_d3zero"]:
        wtmad_1_subset.loc["summary", (data_path_name, f"AI{d3_name.upper()}")] = 0
        wtmad_1_subset.loc["summary", (data_path_name, f"DFT{d3_name.upper()}")] = 0
        wtmad_2_subset.loc["summary", (data_path_name, f"AI{d3_name.upper()}")] = 0
        wtmad_2_subset.loc["summary", (data_path_name, f"DFT{d3_name.upper()}")] = 0
        for name_set in full_subset_dict.keys():
            wtmad_1_subset.loc[
                "summary", (data_path_name, f"AI{d3_name.upper()}")
            ] += wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            wtmad_1_subset.loc[
                "summary", (data_path_name, f"DFT{d3_name.upper()}")
            ] += wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")]
            wtmad_2_subset.loc[
                "summary", (data_path_name, f"AI{d3_name.upper()}")
            ] += wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            wtmad_2_subset.loc[
                "summary", (data_path_name, f"DFT{d3_name.upper()}")
            ] += wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")]

print("Summary")
display(df_summary_subset_ele)
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# save summary to csv with date
df_summary_subset_ele.to_csv(f"../validate_hkqai_done/df_summary_subset_ele_{date}.csv")
df_summary_subset.to_csv(f"../validate_hkqai_done/summary_subset_{date}.csv")
mean_subset.to_csv(f"../validate_hkqai_done/mean_subset_{date}.csv")
wtmad_1_subset.to_csv(f"../validate_hkqai_done/wtmad_1_subset_{date}.csv")
wtmad_2_subset.to_csv(f"../validate_hkqai_done/wtmad_2_subset_{date}.csv")
# save summary to excel with date
df_summary_subset_ele.to_excel(f"../validate_hkqai_done/df_summary_subset_ele_{date}.xlsx")
df_summary_subset.to_excel(f"../validate_hkqai_done/summary_subset_{date}.xlsx")
mean_subset.to_excel(f"../validate_hkqai_done/mean_subset_{date}.xlsx")
wtmad_1_subset.to_excel(f"../validate_hkqai_done/wtmad_1_subset_{date}.xlsx")
wtmad_2_subset.to_excel(f"../validate_hkqai_done/wtmad_2_subset_{date}.xlsx")

cc-pVDZ
103.45767788233127 109.493
197.87474320132537 213.169
67.7890541336356 73.57
301.7467616007382 324.945
264.30226880356827 281.287
78.83721184493066 84.995
177.19211774472035 190.745
66.19061508512556 73.921
492.3781024508485 535.885
286.8025965786612 307.87
394.25825746381474 420.42
566.5643705795687 607.023
340.5786669225199 382.753
218.0213781856794 242.267
666.5164644713718 713.08
940.5028523511942 1007.909
164.31614701852425 181.456
74.9817887194668 84.221
166.15324029093608 183.913
77.9978403703426 87.731
665.7382671656451 721.502
530.9323428577784 582.301
387.86547605063174 422.959
798.0293574686789 861.578
265.7823781808104 298.018
746.5917383465116 811.241
430.9585935350008 474.629
521.6135155617778 564.095
469.62903003021006 513.501
97.68035093485378 107.499
159.49529398107018 182.591
70.88252791147994 83.096
434.5276205401135 482.276
373.77507352331304 410.973
207.5764702942979 232.974
125.78342166014772 141.64
408.5184302926275 446.081
93.69320042856003 107.208
645.2

data_path       1424849                                            \
Ele type  error_scf_ele error_dft_ele error_scf_dip error_dft_dip   
summary         0.32847      0.352175      0.031172       0.03176   
W4_11          0.128193      0.146661      0.023594      0.021439   
G21EA          0.079752      0.090296      0.012511      0.012762   
G21IP          0.081996      0.086805      0.005995      0.006154   
DIPCS10        0.101593      0.107438      0.005017      0.003622   
PA26           0.268845        0.2982      0.034744      0.038429   
SIE4x4         0.071736      0.093996      0.003569      0.001311   
ALKBDE10       0.098409       0.10564      0.065934      0.061711   
YBDE18         0.257196      0.276239      0.041119      0.041601   
AL2X6          0.395256      0.396501      0.002764      0.002336   
HEAVYSB11      0.288568      0.278481      0.011631      0.009043   
NBPRC          0.237834      0.252724      0.032958      0.032482   
ALK8           0.202733      0.187227      0.040614      0.040223   
RC21           0.244294      0.258984      0.071539      0.082447   
G2RC           0.163856      0.181463      0.015502      0.015617   
BH76RC          0.12608       0.14036      0.055976      0.057338   
FH51           0.342554      0.346443      0.024006      0.026834   
TAUT15         0.369822      0.413544      0.067899      0.069453   
DC13           0.390599      0.397856      0.020702      0.019322   
MB16_43        0.500446      0.536587      0.084707      0.086338   
DARC           0.426179      0.443789      0.014191      0.019082   
RSE43          0.209548      0.233351      0.034582      0.033645   
BSR36          0.534747      0.495778      0.004311      0.001683   
CDIE20         0.370491       0.36033      0.039229      0.043548   
ISO34           0.29061      0.298776      0.025594      0.026941   
PArel          0.522926      0.599811      0.074776      0.079046   
BH76            0.12608       0.14036      0.055976      0.057338   
BHPERI         0.329315      0.326909      0.031284      0.029987   
BHDIV10        0.342553      0.361694      0.049657      0.049067   
INV24          0.778226      0.784024      0.038296      0.027518   
BHROT27        0.289291      0.303021      0.021441      0.018507   
PX13           0.215428      0.278637      0.012669       0.01315   
WCPT18         0.197421      0.224154      0.060305      0.057861   
RG18           0.204128      0.206046        0.0023      0.002641   
ADIM6          0.480426      0.434301      0.000826       0.00024   
S22            0.385426      0.405222      0.025222      0.024365   
S66            0.331414      0.347655      0.022744      0.025355   
WATER27        0.393804      0.547345       0.02721      0.033217   
CARBHB12       0.176663      0.195377      0.033396      0.036954   
PNICO23        0.208847      0.233577        0.0258      0.023517   
HAL59          0.399949      0.412527      0.042605      0.036969   
AHB21          0.108228      0.129997      0.032458      0.031459   
CHB6           0.162305       0.16005      0.019153      0.019259   
IL16           0.290156      0.335812      0.049496       0.05021   
IDISP           0.89541      0.816401      0.002265      0.000619   
ICONF          0.464768      0.533359      0.023171      0.022986   
ACONF          0.384884      0.355398      0.003603      0.001781   
Amino20x4      0.723524      0.806256       0.02953      0.037327   
PCONF21         1.05244      1.184712      0.058859      0.077448   
MCONF          0.933391      0.987572      0.038508      0.045477   
SCONF          0.606237      0.725383      0.033587      0.024507   
BUT14DIOL      0.338723      0.380054      0.025886       0.02199   

data_path       1513512                                            \
Ele type  error_scf_ele error_dft_ele error_scf_dip error_dft_dip   
summary        0.322465      0.353225       0.03193       0.03176   
W4_11          0.175284      0.148743       0.02206      0.021439 

MAE


data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1       19.257894   9.561783  19.003664   9.285475  19.001926   9.339163   
sub2        17.43575  17.941953  15.308423  15.070735   14.42948  14.694369   
sub3        3.579974   5.232716   3.500591   5.229833   3.536558   5.240787   
sub4        5.767958    7.73056   5.768147   7.837984   6.029571   8.125751   
sub5        1.877407    1.63367   1.784048   1.215567   1.828415   1.260364   

data_path              1513512                        ...    1428227  \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1           DONE  17.132365   9.561783  16.843939  ...  18.897742   
sub2           DONE  14.708169  17.941917  11.756054  ...  17.406645   
sub3           DONE   3.884964   5.232716   3.871375  ...   4.339606   
sub4           DONE   6.767833    7.73056   6.825807  ...   6.950488   
sub5           DONE   1.537359    1.63367   1.091891  ...   1.427438   

data_path                         3036943                                   \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ   DFT_D3BJ   
sub1        9.242701      DONE  18.138858   9.561783  17.739346   9.285406   
sub2       12.824076      DONE  20.759914  17.941917  17.407069  15.070004   
sub3         5.39491      DONE   4.394932   5.232716   4.194984   5.229844   
sub4        8.966458      DONE   7.589782    7.73056   7.655925   7.838035   
sub5        1.444903      DONE   1.608105    1.63367   1.167015   1.215535   

data_path                                  
Disp type  AI_D3ZERO DFT_D3ZERO Processed  
sub1       17.835187   9.339154      DONE  
sub2       17.292773   14.69417      DONE  
sub3        4.190525   5.240792      DONE  
sub4        7.935859   8.125779      DONE  
sub5        1.203425   1.260358      DONE  

[5 rows x 35 columns]

wtmad_1


data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1         9.04957    8.07025   8.723117   7.669073   8.643578   7.645286   
sub2       10.373674    9.55389    9.58728   8.328621   9.429804   8.192154   
sub3        5.589269    5.26824   5.511509   5.231204    5.58283   5.202779   
sub4       16.212132  15.433344  11.448574   10.51843  11.386519  11.046215   
sub5       16.809623  13.864114   16.57373  10.474968  17.341207  11.075081   
summary    58.034268  52.189838  51.844211  42.222296  52.383938  43.161514   

data_path              1513512                        ...    1428227  \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1           DONE   8.356141    8.07025   7.968809  ...   9.594935   
sub2           DONE   8.294925    9.55386   7.070922  ...   9.809317   
sub3           DONE   3.892173    5.26824   3.880119  ...   5.069737   
sub4           DONE  14.911936  15.433344   9.097337  ...   7.681217   
sub5           DONE  13.017061  13.864114   9.665924  ...   13.02307   
summary          --  48.472236  52.189807  37.683111  ...  45.178276   

data_path                         3036943                                   \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ   DFT_D3BJ   
sub1        7.419066      DONE   9.237687    8.07025   8.672857   7.668977   
sub2        7.363037      DONE   9.462007    9.55386   7.990014   8.328335   
sub3        5.337975      DONE   4.762185    5.26824   4.501345   5.231209   
sub4       12.271066      DONE  15.262996  15.433344   9.168287  10.517844   
sub5       13.286521      DONE  13.873863  13.864114  10.288167  10.474792   
summary    45.677664        --   52.59874  52.189807  40.620669  42.221158   

data_path                                  
Disp type  AI_D3ZERO DFT_D3ZERO Processed  
sub1        8.622622   7.645266      DONE  
sub2        8.014716   8.192048      DONE  
sub3        4.434353   5.202781      DONE  
sub4        9.707163   11.04597      DONE  
sub5       10.853381  11.075089      DONE  
summary    41.632235  43.161155        --  

[6 rows x 35 columns]

wtmad_2


data_path    1424849                                                        \
Disp type         AI       DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        6.277319  5.178214   6.172154   5.078621   6.164673   5.079784   
sub2        3.651251  3.877689   3.242347   3.272638   3.123791   3.196425   
sub3        2.106216  2.614724   2.063461   2.616758   2.085172   2.626665   
sub4        6.367586  6.373283   4.862385   4.749006   4.850364   4.934288   
sub5        8.814703  6.885461   8.927999   5.505604   9.303028   5.735238   
summary    27.217075  24.92937  25.268346  21.222626  25.527028    21.5724   

data_path              1513512                        ...    1428227  \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1           DONE    5.94125   5.178214   5.866193  ...   6.560451   
sub2           DONE   3.561736   3.877663   2.940609  ...    3.25987   
sub3           DONE   1.980558   2.614724   1.969746  ...    2.16422   
sub4           DONE   6.171102   6.373283   4.261619  ...   3.854685   
sub5           DONE   6.413561   6.885461   4.959104  ...   6.412078   
summary          --  24.068207  24.929344  19.997271  ...  22.251304   

data_path                         3036943                                   \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ   DFT_D3BJ   
sub1        5.034329      DONE   6.238372   5.178214   6.061484   5.078599   
sub2        2.768723      DONE   3.731224   3.877663   3.037659   3.272491   
sub3        2.712063      DONE   2.260037   2.614724   2.161454   2.616764   
sub4        5.657324      DONE   6.361915   6.373283   4.359116     4.7488   
sub5        7.087473      DONE   6.706495   6.885461   5.172812   5.505572   
summary    23.259912        --  25.298042  24.929344  20.792525  21.222225   

data_path                                 
Disp type AI_D3ZERO DFT_D3ZERO Processed  
sub1       6.034555   5.079779      DONE  
sub2       3.014919   3.196361      DONE  
sub3       2.159593   2.626668      DONE  
sub4       4.578321   4.934221      DONE  
sub5       5.380061   5.735267      DONE  
summary    21.16745  21.572295        --  

[6 rows x 35 columns]

Summary of Subset
MAE


data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
W4_11      39.203149  10.586851  38.853509  10.151262  38.979171   10.36342   
G21EA      30.220527  21.305991  30.220361  21.305565  30.218268  21.312705   
G21IP       8.904852   4.048451   8.906119   4.050199   8.907709   4.051561   
DIPCS10    15.645142   5.523671  15.642432   5.520994  15.670327   5.542695   
PA26        5.547839   4.345868   5.734141   4.560076   5.766502   4.551628   
SIE4x4      2.211917  19.742582   2.344023  19.931231   2.441618  19.995962   
ALKBDE10   24.646299   7.329434  24.542879   7.226645  24.639264   7.322383   
YBDE18     17.630289  11.624199  16.612456  10.271846  16.195151  10.131819   
AL2X6       8.563034   8.882512   7.106967   6.977665   6.724394   6.882542   
HEAVYSB11   13.67463   9.234585  12.807459   8.064498  12.613713   8.085818   
NBPRC       4.818544   3.439714     4.2556   2.450943   3.931592   2.304782   
ALK8        6.646991   5.808698   5.861682   4.687731   5.453268   4.528804   
RC21         3.96282   3.461899   3.375288   4.132064   3.184984   4.320594   
G2RC        7.470471   9.046991   7.521257   9.158742   7.542723   9.175369   
BH76RC     19.743386  19.296993  19.769568   19.33919  19.808283  19.373516   
FH51        4.914566   5.989218   4.693241   5.778463   4.710783    5.71526   
TAUT15      2.047229   1.296414   2.056397   1.293938   2.107852   1.297826   
DC13       13.168067  11.283548   12.51633   9.855071  12.284805    9.62655   
MB16_43    70.165924  69.184937  62.479214  59.061723  58.869199   57.48081   
DARC        6.458438   9.156028   4.510171   6.415948   3.932408   6.158905   
RSE43       2.192437     2.1819   2.124583   2.076916   2.001252   1.920009   
BSR36       7.722201  11.536749   5.448674   8.305676    5.06645   8.366058   
CDIE20      1.389784    1.30392   1.382773   1.210818   1.403786   1.121308   
ISO34       2.600893   2.542523   2.498489   2.371063   2.408512   2.308607   
PArel       3.272731   2.150234   3.245339    2.11164   3.267522   2.163034   
BH76        5.081367   8.001229   5.039622   8.263441   5.077368   8.336214   
BHPERI       2.31662   2.267963   1.772904   1.473761   1.804448   1.476689   
BHDIV10     2.804841   3.632748   2.755641   3.770441   2.683666   3.713918   
INV24       3.496892   2.365007   3.596044   1.874413   3.568984   1.786513   
BHROT27      2.02935   0.460483   2.033646   0.457262   2.061739   0.452308   
PX13        2.207445   9.108108   2.039994   9.347137   2.153157    9.19119   
WCPT18      2.924216   6.897796   3.039895   7.316616   3.174801   7.391851   
RG18        0.718153   0.639009   0.530815     0.4001   0.473066   0.375029   
ADIM6       5.078231   4.538994   2.512195   1.504767   2.325509   1.475628   
S22          3.25348    2.96243   2.400718   1.818909   2.450278   1.921899   
S66         2.719507   2.650196    1.70671   1.613102   1.701413   1.728748   
WATER27    26.776433  44.026732  30.374405  48.436778  32.080143  49.886845   
CARBHB12    1.107515   1.225261    0.93317   1.720435   0.849189   1.838808   
PNICO23     1.247076   1.360456   0.989045    1.11031    1.09178   1.251345   
HAL59        2.16396   1.760779   1.907658    1.44248   1.998212   1.549255   
AHB21       6.674573   8.209303   6.876886   8.591839   7.042015   8.737606   
CHB6        5.328572   4.975051   5.327903   5.211979   5.140394   5.355778   
IL16        7.734363   8.085039   8.889882   9.542067   9.668163  10.191702   
IDISP      14.868039  14.656005  10.125663   8.360211   9.397027   8.255014   
ICONF       0.939276   0.436156   0.899487   0.401956    0.84748   0.496735   
ACONF       3.231744   0.870997   2.996062   0.444413   2.945497   0.414074   
Amino20x4   1.327476   0.667886    1.30662   0.588023   1.325975   0.630334   
PCONF21     1.393949   1.543614   1.871713   0.774716   2.345268   0.892306   
MCONF        1.39839   1.627101   1.178519   0.355

In [1]:
i_reaction

NameError: name 'i_reaction' is not defined